In [1]:
#!pip install selenium

import re
import html

from selenium import webdriver
from selenium.webdriver.common.keys import Keys 

from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

import time
import locale
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [3]:
# from google.colab import drive
# drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [4]:
#cd /content/gdrive/My Drive/Colab Notebooks

/content/gdrive/My Drive/Colab Notebooks


In [2]:
# functions to optimize code
def split_level_type_duty_trade(words, lst, Text):
    
    if words in Text:
        Text = Text.replace(words,'')
        Text = Text.replace('\n','')
        lst.append(Text)
    else:
        lst.append('-')
        
    return lst


In [3]:
def LinkedIn_webCrawl(scrolls, keywords):

    jobTitle = []
    jobCompany = []
    address = []
    builtIn = []
    jobDetail_URL = []
    content=[]
    level=[]
    type=[]
    duty=[]
    trade=[]

    # bs4 settings
    HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36",
              'cookie': 'over18=1;'}

    # selenium settings

    options=Options()
    options.chrome_executable_path='./chromedriver'
    driver=webdriver.Chrome(options=options)
    #driver.get('https://www.linkedin.com/jobs/search/?location=%E5%8F%B0%E7%81%A3&geoId=104187078&f_TPR=r86400&f_JT=F&currentJobId=3482773339&position=1&pageNum=0')
    driver.get(f'https://www.linkedin.com/jobs/search/?keywords={keywords}&location=%E5%8F%B0%E7%81%A3&locationId=&geoId=104187078&f_TPR=r604800&f_JT=F&position=1&pageNum=0')

    # total jobs count
    jobs_num = driver.find_element(By.CLASS_NAME, 'results-context-header__job-count')
    locale.setlocale( locale.LC_ALL, 'en_US.UTF-8' ) 
    jobs_num = locale.atoi(jobs_num.text)
    scrolls_max = jobs_num//25
    
    scroll=0
    
    if scrolls>scrolls_max:
        scrolls=scrolls_max
        
        print(f"Max scroll down counts: {scrolls}")

    while scroll<=scrolls:

        if scrolls>=6:
            jobMoreBtn = driver.find_elements(By.TAG_NAME, 'button')

            try:
                for btn in jobMoreBtn[::-1]:
                    if btn.text=='更多職缺': # or btn.text=='Show more':
                        jobMoreBtn = btn

                        break

                jobMoreBtn.send_keys(Keys.ENTER)

                driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
                time.sleep(3)

                scroll+=1

            except:
                driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
                time.sleep(3)
                scroll+=1
        
        else:
          #if scroll<=5:
            driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
            time.sleep(3)
            scroll+=1

    titleTags=driver.find_elements(By.CLASS_NAME, 'base-search-card__title')
    companyTags=driver.find_elements(By.CLASS_NAME, 'base-search-card__subtitle')
    workcityTags=driver.find_elements(By.CLASS_NAME, 'job-search-card__location')
    onlineTags=driver.find_elements(By.TAG_NAME, 'time')
    jobDetailURL=driver.find_elements(By.TAG_NAME, 'a')

    for n in range(len(titleTags)):
        jobTitle.append(titleTags[n].text)
        jobCompany.append(companyTags[n].text)
        address.append(workcityTags[n].text)
        builtIn.append(onlineTags[n].get_attribute('datetime'))


    for n in range(len(jobDetailURL)):

        j = jobDetailURL[n].get_attribute('href')
        if 'jobs/view' in j and 'result_search-card' in j:
            jobDetail_URL.append(j)


    for url in jobDetail_URL:
        r = requests.get(url, headers=HEADERS)
        r.encoding = 'utf-8' #轉換編碼至UTF-8
        #soup = BeautifulSoup(r.text, features='lxml')
        soup = BeautifulSoup(r.text)

        jobContent = soup.find("section",'show-more-less-html')
        try:
            content_text = jobContent.text
            content_text = content_text.replace('\n','')
            content.append(content_text)
        except:
            content.append('-')

        jobItems = soup.find_all("li",'description__job-criteria-item')

        if len(jobItems)==4:
            for i in jobItems:
                Text = i.text
                if '職階等級' in Text:
                    Text = Text.replace('職階等級','')
                    Text = Text.replace('\n','')
                    level.append(Text)
                elif '工作類型' in Text:
                    Text = Text.replace('工作類型','')
                    Text = Text.replace('\n','')
                    type.append(Text)
                elif '工作職能' in Text:
                    Text = Text.replace('工作職能','')
                    Text = Text.replace('\n','')
                    duty.append(Text)
                elif '所屬產業' in Text:
                    Text = Text.replace('所屬產業','')
                    Text = Text.replace('\n','')
                    trade.append(Text)
        else:
            Text = i.text
            level = split_level_type_duty_trade('職階等級', level, Text)
            type = split_level_type_duty_trade('工作類型', type, Text)
            duty = split_level_type_duty_trade('工作職能', duty, Text)
            trade = split_level_type_duty_trade('所屬產業', trade, Text)

    driver.close()

    df = pd.DataFrame()

    df['職稱']=jobTitle
    df['公司']=jobCompany
    df['地址']=address
    df['職缺內容']=content
    df['職缺上架日期']=builtIn
    df['職等']=level
    df['工作類型']=type
    df['職務']=duty
    df['產業']=trade

    return df



In [4]:
test = LinkedIn_webCrawl(2,'Data Scientist')
test

,職稱,公司,地址,職缺內容,職缺上架日期,職等,工作類型,職務,產業
0,Principal Data Scientist,台積電,新竹市,Roles and ResponsibilitiesAs an AI/ML Manager ...,2023-11-08,中高階,全職,分析師、企劃和IT ...,家電、電器與電子產品製造和半...
1,Product Developer (影像分析應用開發),Synology,台北,Department Info Synology's product developers ...,2023-11-09,初階,全職,產品管理和行銷 ...,IT 服務與 IT 諮詢 ...
2,Security Data Scientist,台積電,新竹市,Responsibility: The Data Scientist will be res...,2023-11-08,中高階,全職,工程 ...,半導體製造 ...
3,AI Software Platform Engineer (Multiple Levels),美商高通國際股份有限公司,台北,CompanyQualcomm Semiconductor LimitedJob AreaE...,2023-11-10,其他,全職,其他 ...,電信通訊 ...
4,AI Software Platform Engineer (Multiple Levels),美商高通國際股份有限公司,Hsinchu City,CompanyQualcomm Semiconductor LimitedJob AreaE...,2023-11-10,其他,全職,其他,電信通訊
...,...,...,...,...,...,...,...,...,...
95,【2024 New College Grads】IT Software Engineer –...,美光科技,Taichung City,Our vision is to transform how the world uses ...,2023-11-08,其他,全職,其他 ...,電腦硬體製造、家...
96,运维工程师,MOLOR PRO PTE. LTD.,"臺北市, 台灣",岗位职责： 1、参与容器平台的架构设计和开发，提升系统稳定性和交付效率 2、研发基础服务组件...,2023-11-13,中高階,全職,金融 ...,軟體開發和技術、...
97,Field Service Engineer,Entegris,臺南市,Company Overview And ValuesWhy work at Entegri...,2023-11-10,初階,全職,IT ...,半導體製造 ...
98,System Test and Application Engineer-WoS (Taipei),美商高通國際股份有限公司,台北,CompanyQualcomm Communication Technologies Ltd...,2023-11-10,其他,全職,其他 ...,電信通訊 ...


In [7]:
test = test.drop_duplicates()

test2 = test[test['職務']=='-']

test2.shape

(33, 9)

In [20]:
# 中英分離

demand = test.filter(['職稱','職務','職缺內容'])

# 创建一个新的DataFrame来存放拆分后的行
new_rows = []

# 遍历原始DataFrame
for index, row in demand.iterrows():
    job_titles = row['職稱'].split('、')
    jobName = row['職務']
    JD = row['職缺內容']
    for title in job_titles:
        new_row = {'職稱': title, '職務': jobName, '職缺描述':JD}
        new_rows.append(new_row)

# 创建包含新行的新DataFrame
new_demand = pd.DataFrame(new_rows)

# data_duty = ['數據分析師','資料科學家','資料工程師','AI工程師','演算法工程師']
# mask = new_demand['職務'].isin(data_duty)
# new_demand = new_demand[mask]

# 將 工作敘述 的網址取代掉
def remove_url_and_html(text):
    # 用 re 取代 URL
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    
    # 用 re 移除 HTML tag
    text = re.sub(r'<.*?>', '', text)
    text = text.replace('\n', '').replace('\r', '').replace('\t', '')
    
    return text

# use re to replace html entities
def replace_html_entities(text):
    
    text = re.sub(r'<.*?>', '', text)
    
    # parse entities
    text = html.unescape(text)
    
    return text

new_demand['職缺描述'] = new_demand['職缺描述'].apply(remove_url_and_html)
new_demand['職缺描述'] = new_demand['職缺描述'].apply(replace_html_entities)

# 中英文 分離
# 提取中文
def extract_chinese(text):
    chinese_pattern = r'[\u4e00-\u9fa5]+'  # 中文字符
    chinese_matches = re.findall(chinese_pattern, text)
    return ' '.join(chinese_matches)  # 將多個中文字符，以空格分隔

# 提取英文
def extract_english(text):
    english_pattern = r'[a-zA-Z]+'  # 英文字符
    english_matches = re.findall(english_pattern, text)
    return ' '.join(english_matches)  # 將多個英文字符，以空格分隔


new_demand['職缺描述（中文）'] = new_demand['職缺描述'].apply(extract_chinese)
new_demand['職缺描述（英文）'] = new_demand['職缺描述'].apply(extract_english)

strings_to_replace = ['工作說明', '工作內容', '職缺投遞', '加分項目', '需求條件', '展開 收回', '職缺說明', '應徵資格', '其他資訊']  # 在这里添加需要替换的字符串
new_demand['職缺描述（中文）'] = new_demand['職缺描述（中文）'].str.replace('|'.join(strings_to_replace), '', regex=True)

new_demand


,職稱,職務,職缺描述,職缺描述（中文）,職缺描述（英文）
0,Artificial Intelligence Research Intern - Deep...,其他 ...,NVIDIA is searching for several outsta...,,NVIDIA is searching for several outstanding co...
1,Senior MLOps Engineer,-,Job OverviewOur team is currently in the early...,,Job OverviewOur team is currently in the early...
2,Solution Engineer,其他 ...,Almost 200 countries. 1 common goal. Making re...,負責弱電安防工程系統測試並協調管理分包技術人員施工品質 負責弱電安防系統架構規劃及設計及現場...,Almost countries common goal Making real what ...
3,Robotics Software Development Engineer (機器人軟體研...,工程 ...,工作說明Are you fascinated by the infinite possibi...,,Are you fascinated by the infinite possibiliti...
4,AI Algorithms SW Engineer - New College Graduate,工程,NVIDIA is searching for Deep Learning ...,,NVIDIA is searching for Deep Learning algorith...
...,...,...,...,...,...
95,EUV Mask & Pellicle Researcher,-,-,,
96,"Software Engineer, Embedded System, Security, ...",IT和工程 ...,Google welcomes people with disabiliti...,,Google welcomes people with disabilities Note ...
97,Biostatistician I,-,-,,
98,Staff/Senior Framework Software Engineer,工程、IT和...,Neuchips cooperation strives to be one of the ...,,Neuchips cooperation strives to be one of the ...


In [21]:
for jd in new_demand['職缺描述（中文）']:
    print(jd)
    print('----------------')


----------------

----------------
負責弱電安防工程系統測試並協調管理分包技術人員施工品質 負責弱電安防系統架構規劃及設計及現場故障查修排除 參加專案工程現場介面相關會議 負責客戶教育訓練及維持高度客戶滿意度 團隊合作解決專案遇到各項問題 大專以上 電子 電機 資訊 工管及企管相關 年以上相關工作經驗問題分析及解決能力良好的溝通及語言表達能力具備文件管理及文書處理能力 
----------------
  
----------------

----------------
  
----------------

----------------
 生產系統資料蒐集 如生產機台設備及製造相關資訊 生產環境控制相關資訊 物料保存環境相關資訊 並使用開發應用軟體呈現 並撰寫設計規格文件 使用 等工具 開發 具有資料庫軟體應用能力 訂定關聯式資料庫規格 以供生產單位了解相關生產細節 讓生產單位維持運作順利 檢視是否有誤並快速解決問題 進行模組化系統設計及軟體整合測試 開發精密的控制系統及 智慧型人機介面  相關設計經驗 年以上  
----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------

----------------
公司簡介龍亭新技股份有限公司 前身為奇美集團之電子紙 事業單位 於 年分割為一獨立公司 龍亭新技為電子紙面板領導廠商 其

In [6]:
test.to_excel('LinkedIn.xlsx',index=False)

In [ ]:
# There should be a terminal window running Jupyter. 
# Navigate to that terminal window and press Control-Z. 
# It will suspend the job and put it in the background. 
# To restart, type 'fg' in the terminal window to bring the job to the foreground.

In [ ]:
#         if scroll<=6:
#             driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
#             time.sleep(3)
#             scroll+=1
#         else:
#             # 點選載入更多
#             jobMoreBtn = driver.find_elements(By.TAG_NAME, 'button')
#             for btn in jobMoreBtn[::-1]:
#                 if btn.text=='更多職缺':
#                     jobMoreBtn = btn

#                     break

#             jobMoreBtn.send_keys(Keys.ENTER)

#             driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
#             time.sleep(3)

#             scroll+=1